# Cryptography Lab Assignment 10
## MT19937 Stream Cipher and Predictable Token Attack

**Course:** Cryptography Laboratory (22CPP307)  
**Group:** Group 07  
**Authors:**  
- **Kunal Sachan** (Roll No: `2024ucp1547`, Email: `2024ucp1547@mnit.ac.in`)  
- **Yashwant Gour** (Roll No: `2024ucp1732`, Email: `2024ucp1732@mnit.ac.in`)  

---
### Objective
Understand why statistical PRNGs (specifically MT19937) are unsuitable for cryptographic applications by implementing a toy stream cipher, recovering a weak 16-bit seed via known-plaintext brute force, and predicting Unix timestamp-seeded password reset tokens.

## Part A — MT19937 Stream Cipher & 16-Bit Seed Recovery
In this section, we implement an 8-bit keystream generator using Python's built-in MT19937 PRNG, construct a symmetric XOR stream cipher, and break a 16-bit key using a known-plaintext substring attack.

In [1]:
import random
import time
import string

def generate_keystream(seed: int, length: int) -> bytes:
    """Generate an 8-bit keystream of specified length using MT19937."""
    rng = random.Random(seed)
    return bytes(rng.getrandbits(8) for _ in range(length))

def mt19937_xor_cipher(data: bytes, seed: int) -> bytes:
    """Symmetric XOR stream cipher (functions as both encrypt and decrypt)."""
    keystream = generate_keystream(seed, len(data))
    return bytes(d ^ k for d, k in zip(data, keystream))

print("[+] MT19937 Stream Cipher functions loaded successfully.")

In [2]:
# Step 2: Verify encryption and decryption using a test message
test_msg = b"CryptoLabX Group 07: Stream Cipher Verification Message 2026!"
test_seed = 0xC001

ciphertext = mt19937_xor_cipher(test_msg, test_seed)
decrypted = mt19937_xor_cipher(ciphertext, test_seed)

print(f"Original : {test_msg.decode()}")
print(f"Ciphertext (Hex): {ciphertext.hex()}")
print(f"Decrypted: {decrypted.decode()}")
assert decrypted == test_msg, "Decryption failed!"
print("[+] Roundtrip Encryption/Decryption Verified: SUCCESS")

In [3]:
# Step 3: Generate plaintext with random prefix of 5-20 characters + 14 consecutive 'A's
prefix_len = random.randint(5, 20)
prefix = "".join(random.choices(string.ascii_letters + string.digits, k=prefix_len)).encode()
known_suffix = b"A" * 14
target_plaintext = prefix + known_suffix

# Encrypt using a randomly selected 16-bit seed (0 to 65535)
actual_seed = random.randint(0, 0xFFFF)
target_ciphertext = mt19937_xor_cipher(target_plaintext, actual_seed)

print(f"Prefix: {prefix} (len {len(prefix)})")
print(f"Known Suffix: {known_suffix}")
print(f"Full Plaintext: {target_plaintext}")
print(f"Secret 16-bit Seed: 0x{actual_seed:04X} ({actual_seed})")
print(f"Ciphertext (Hex): {target_ciphertext.hex()}")

In [4]:
# Step 4 & 5: Brute-force all 65,536 seeds to recover matching seed and plaintext
start_time = time.perf_counter()
recovered_seed = None
recovered_plaintext = None
total_seeds = 65536

for candidate_seed in range(total_seeds):
    rng = random.Random(candidate_seed)
    ks = bytes(rng.getrandbits(8) for _ in range(len(target_ciphertext)))
    pt = bytes(c ^ k for c, k in zip(target_ciphertext, ks))
    if known_suffix in pt:
        recovered_seed = candidate_seed
        recovered_plaintext = pt
        break

elapsed = time.perf_counter() - start_time
print(f"Search completed in {elapsed:.4f} seconds")
print(f"Recovered Seed: 0x{recovered_seed:04X} ({recovered_seed})")
print(f"Actual Seed   : 0x{actual_seed:04X} ({actual_seed})")
print(f"Decrypted Text: {recovered_plaintext}")
assert recovered_seed == actual_seed
assert recovered_plaintext == target_plaintext
assert known_suffix in recovered_plaintext
print("[+] Part A Brute-Force & Plaintext Verification: ALL PASSED!")

## Part B — Predictable Password Reset Tokens & Interval Timestamp Attack
In this section, we simulate a web application that generates 16-byte password reset tokens using MT19937 seeded with the current Unix epoch integer timestamp $T$. We then simulate an attacker who intercepts the token and predicts the seed within a $[T-300, T+300]$ seconds window.

In [5]:
# Step 6, 7 & 8: Token generation using current Unix timestamp
def generate_reset_token(t: int) -> bytes:
    rng = random.Random(t)
    return bytes(rng.getrandbits(8) for _ in range(16))

actual_T = int(time.time())
token_bytes = generate_reset_token(actual_T)
token_hex = token_bytes.hex()

print(f"Timestamp T (Unix seconds): {actual_T}")
print(f"Formatted UTC Date/Time   : {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime(actual_T))}")
print(f"Generated 16-byte Token   : {token_hex}")

In [6]:
# Step 9, 10 & 11: Token prediction within [T-300, T+300] window
approx_time = actual_T + 25  # Attacker tests 25 seconds later
window = 300
start_ts = approx_time - window
end_ts = approx_time + window

start_bench = time.perf_counter()
recovered_T = None
tested_count = 0

for candidate_t in range(start_ts, end_ts + 1):
    tested_count += 1
    candidate_token = generate_reset_token(candidate_t)
    if candidate_token == token_bytes:
        recovered_T = candidate_t
        break

search_time = time.perf_counter() - start_bench
print(f"Search Interval       : [{start_ts}, {end_ts}] ({end_ts - start_ts + 1} candidates)")
print(f"Candidates Tested     : {tested_count}")
print(f"Elapsed Search Time   : {search_time*1000:.3f} ms")
print(f"Recovered Timestamp   : {recovered_T}")
print(f"Actual Timestamp      : {actual_T}")

# Verify regeneration
regen_token = generate_reset_token(recovered_T)
assert regen_token == token_bytes
print(f"Regenerated Token Hex : {regen_token.hex()}")
print("[+] Part B Token Prediction & Regeneration Match: VERIFIED SUCCESS!")

## Notebook Work: Answers to Questions 12 – 15

### 12. MT19937 Algorithm and Why It Is Not Cryptographically Secure
- **Mathematical Structure:** MT19937 is a linear feedback shift register operating over $\mathrm{GF}(2)$ with a period of $2^{19937}-1$. The state consists of 624 32-bit unsigned integers (19,937 bits).
- **Linearity:** The state recurrence and output tempering functions are completely linear over $\mathbb{F}_2$ (consisting entirely of bitwise XOR, AND, and shifts).
- **State Recovery (Untempering):** Because tempering is a bijection with elementary bitwise inverses, observing 624 consecutive 32-bit outputs allows an adversary to invert the tempering transformations, recover all 624 internal words, and predict all future and past outputs with 100% certainty.
- **Failure of Next-Bit Test:** It is not a CSPRNG and fails polynomial-time unpredictability.

### 13. Keystream Revelation from Known Plaintext & 16-Bit Seed Search Space
- **XOR Stream Cipher Equation:** $C = P \oplus K \implies K = C \oplus P$. Any known plaintext revealed immediately yields the keystream bytes.
- **16-Bit Search Space:** $2^{16} = 65,536$ candidate seeds ($16$ bits of entropy). Modern processors evaluate the entire space in less than 0.5 seconds, offering negligible computational resistance.

### 14. Vulnerability of Timestamp-Based PRNG Seeding
- **Low Entropy:** A $\pm 300$ second window contains only $601$ candidates ($\approx 9.23$ bits of entropy).
- **Security Impact:** Attackers can trivially reconstruct password reset tokens, session identifiers, or encryption keys in milliseconds, resulting in complete account takeover.

### 15. Cryptographically Secure Recommendations
- Use OS-level CSPRNGs (`/dev/urandom`, Python `secrets` module, `os.urandom`).
- Ensure minimum 128-bit true entropy for security-sensitive tokens and keys.